# Optional: Testing, Troubleshooting & Cleanup

## Agentic Security Hands-On Lab

**Estimated Time:** 20-30 minutes

---

This notebook provides commands and procedures for:
- Testing infrastructure components
- Troubleshooting common issues
- Cleaning up resources
- Docker vs Podman considerations

## 🧪 Testing Infrastructure

Use these commands to verify that all services are running correctly.

### Check Service Status

In [ ]:
import subprocess
import os
from pathlib import Path

# Get lab root directory
lab_root = Path.cwd().parent
os.chdir(lab_root)

print("📊 Checking service status...\n")
result = subprocess.run(['podman-compose', 'ps'], capture_output=True, text=True)
print(result.stdout)

print("\n💡 All services should show 'Up' or 'healthy' status")

### Test Vault Connectivity

In [ ]:
import requests

print("🔐 Testing Vault connectivity...\n")

try:
    response = requests.get('http://localhost:8200/v1/sys/health', timeout=5)
    if response.status_code == 200:
        health = response.json()
        print("✅ Vault is healthy!")
        print(f"   Initialized: {health.get('initialized')}")
        print(f"   Sealed: {health.get('sealed')}")
        print(f"   Version: {health.get('version')}")
    else:
        print(f"⚠️  Vault returned status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"❌ Cannot connect to Vault: {e}")
    print("   Make sure Vault container is running")

### Test PostgreSQL Connectivity

In [ ]:
print("🐘 Testing PostgreSQL connectivity...\n")

result = subprocess.run(
    ['podman', 'exec', '-it', 'agentic-postgres', 
     'psql', '-U', 'postgres', '-d', 'employee_db', 
     '-c', 'SELECT COUNT(*) FROM employee_basic_info;'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ PostgreSQL is accessible!")
    print(result.stdout)
else:
    print("❌ Cannot connect to PostgreSQL")
    print(result.stderr)

### Test Langflow

In [ ]:
print("🤖 Testing Langflow...\n")

try:
    response = requests.get('http://localhost:7860/health', timeout=5)
    if response.status_code == 200:
        print("✅ Langflow is healthy!")
        print(f"   Response: {response.json()}")
    else:
        print(f"⚠️  Langflow returned status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"❌ Cannot connect to Langflow: {e}")
    print("   Make sure Langflow container is running")

### Test AskHR Backend

In [ ]:
print("🌐 Testing AskHR Backend...\n")

try:
    response = requests.get('http://localhost:5001/health', timeout=5)
    if response.status_code == 200:
        print("✅ AskHR Backend is healthy!")
        print(f"   Response: {response.json()}")
    else:
        print(f"⚠️  AskHR Backend returned status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"❌ Cannot connect to AskHR Backend: {e}")
    print("   Make sure backend container is running")

### Verify Vault Configuration

In [ ]:
import os

print("🔍 Verifying Vault configuration...\n")

# Set Vault environment variables
os.environ['VAULT_ADDR'] = 'http://localhost:8200'
os.environ['VAULT_TOKEN'] = 'root'

# List policies
print("📋 Vault Policies:")
result = subprocess.run(['vault', 'policy', 'list'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print(f"❌ Error: {result.stderr}")

# List database roles
print("\n🗄️  Database Roles:")
result = subprocess.run(['vault', 'list', 'database/roles'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print(f"❌ Error: {result.stderr}")

# List identity groups
print("\n👥 Identity Groups:")
result = subprocess.run(['vault', 'list', 'identity/group/name'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print(f"❌ Error: {result.stderr}")

---

## 🐛 Troubleshooting

Common issues and their solutions.

### Check Container Logs

When a service is not working, check its logs:

In [ ]:
# Choose which service to check (uncomment one)
service_name = 'vault'  # Options: vault, postgres, langflow, askhr-backend, askhr-frontend

print(f"📜 Checking logs for {service_name}...\n")
result = subprocess.run(
    ['podman-compose', 'logs', '--tail=50', service_name],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("\nErrors:")
    print(result.stderr)

### Restart a Specific Service

In [ ]:
# Choose which service to restart
service_name = 'vault'  # Options: vault, postgres, langflow, askhr-backend, askhr-frontend

print(f"🔄 Restarting {service_name}...\n")
result = subprocess.run(
    ['podman-compose', 'restart', service_name],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✅ {service_name} restarted successfully")
else:
    print(f"❌ Error restarting {service_name}")
    print(result.stderr)

### Restart All Services

In [ ]:
print("🔄 Restarting all services...\n")
result = subprocess.run(
    ['podman-compose', 'restart'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ All services restarted successfully")
    print(result.stdout)
else:
    print("❌ Error restarting services")
    print(result.stderr)

### Re-apply Vault Configuration (Terraform)

In [ ]:
print("🔧 Re-applying Vault configuration with Terraform...\n")

terraform_dir = lab_root / 'infrastructure' / 'vault' / 'terraform'
os.chdir(terraform_dir)

result = subprocess.run(
    ['terraform', 'apply', '-auto-approve'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Terraform apply completed successfully")
    print(result.stdout[-500:])  # Show last 500 chars
else:
    print("❌ Terraform apply failed")
    print(result.stderr)

# Change back to lab root
os.chdir(lab_root)

### Check Database Tables

In [ ]:
print("🗄️  Checking database tables...\n")

result = subprocess.run(
    ['podman', 'exec', '-it', 'agentic-postgres',
     'psql', '-U', 'postgres', '-d', 'employee_db',
     '-c', '\\dt'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Database tables:")
    print(result.stdout)
else:
    print("❌ Error accessing database")
    print(result.stderr)

### Common Issues and Solutions

#### Issue: Services Not Starting
**Solution:**
1. Check if Podman machine is running: `podman machine list`
2. Start Podman machine if needed: `podman machine start`
3. Check port conflicts: `lsof -i :8200` (Vault), `lsof -i :5432` (PostgreSQL), etc.
4. Review logs: `podman-compose logs <service-name>`

#### Issue: Vault Configuration Not Applied
**Solution:**
1. Verify Vault is unsealed: `vault status`
2. Check Terraform state: `cd infrastructure/vault/terraform && terraform show`
3. Re-apply configuration: `terraform apply`

#### Issue: Database Connection Errors
**Solution:**
1. Verify PostgreSQL is running: `podman ps | grep postgres`
2. Check database initialization: `podman logs agentic-postgres`
3. Test connection: `podman exec -it agentic-postgres psql -U postgres -d employee_db`

#### Issue: Langflow Not Accessible
**Solution:**
1. Check Langflow logs: `podman-compose logs langflow`
2. Verify Langflow database: `podman ps | grep langflow-postgres`
3. Restart Langflow: `podman-compose restart langflow`
4. Wait for initialization (can take 1-2 minutes)

#### Issue: Frontend Authentication Errors
**Solution:**
1. Verify IBM Verify credentials in `.env` file
2. Confirm redirect URI is registered: `http://localhost:3000/callback`
3. Check backend logs: `podman-compose logs askhr-backend`
4. Verify CORS settings in backend

#### Issue: Langflow Chat Widget Not Working
**Solution:**
1. Verify `REACT_APP_LANGFLOW_API_KEY` in `.env`
2. Confirm `REACT_APP_LANGFLOW_FLOW_ID` matches your flow
3. Check `REACT_APP_LANGFLOW_COMPONENT_ID` is correct
4. Test Langflow health: `curl http://localhost:7860/health`
5. Check browser console for errors

---

## 🧹 Cleanup

Commands to clean up resources after completing the lab.

### Stop All Services (Keep Data)

In [ ]:
print("🛑 Stopping all services...\n")

os.chdir(lab_root)
result = subprocess.run(
    ['podman-compose', 'down'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ All services stopped")
    print(result.stdout)
else:
    print("❌ Error stopping services")
    print(result.stderr)

print("\n💡 Data volumes are preserved. Use 'podman-compose up -d' to restart.")

### Stop All Services and Remove Volumes (Delete All Data)

⚠️ **WARNING:** This will delete all data including database contents and Vault data!

In [ ]:
print("⚠️  WARNING: This will delete ALL data!\n")
print("🗑️  Stopping services and removing volumes...\n")

os.chdir(lab_root)
result = subprocess.run(
    ['podman-compose', 'down', '-v'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ All services stopped and volumes removed")
    print(result.stdout)
else:
    print("❌ Error during cleanup")
    print(result.stderr)

print("\n💡 All data has been deleted. You'll need to reconfigure everything if you restart.")

### Destroy Vault Configuration (Terraform)

In [ ]:
print("🔥 Destroying Vault Terraform configuration...\n")

terraform_dir = lab_root / 'infrastructure' / 'vault' / 'terraform'
os.chdir(terraform_dir)

result = subprocess.run(
    ['terraform', 'destroy', '-auto-approve'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Terraform destroy completed")
    print(result.stdout[-500:])  # Show last 500 chars
else:
    print("❌ Terraform destroy failed")
    print(result.stderr)

os.chdir(lab_root)

### Remove All Podman Images (Optional)

In [ ]:
print("🗑️  Removing Podman images used in this lab...\n")

images = [
    'hashicorp/vault:latest',
    'postgres:15',
    'langflowai/langflow:latest'
]

for image in images:
    print(f"Removing {image}...")
    result = subprocess.run(
        ['podman', 'rmi', image],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"  ✅ Removed {image}")
    else:
        print(f"  ⚠️  Could not remove {image} (may not exist or be in use)")

print("\n💡 Images removed. They will be re-downloaded if you restart the lab.")

### Deactivate Python Virtual Environment

Run this in your terminal (not in the notebook):

```bash
deactivate
```

---

## 🐳 Docker vs Podman

This lab uses Podman by default, but you can use Docker instead. Here's what you need to know.

### Key Differences

| Feature | Podman | Docker |
|---------|--------|--------|
| **Daemon** | Daemonless (runs as user process) | Requires Docker daemon |
| **Root Access** | Can run rootless | Typically requires root/sudo |
| **Security** | More secure by default | Requires additional configuration |
| **Compatibility** | Docker-compatible CLI | Native Docker |
| **Compose** | Requires `podman-compose` | Native `docker-compose` |
| **Kubernetes** | Built-in pod support | Requires additional tools |

### Using Docker Instead of Podman

If you prefer to use Docker, make these changes:

#### 1. Replace Commands

Replace all `podman` commands with `docker`:

```bash
# Podman
podman-compose up -d
podman ps
podman logs <container>

# Docker
docker-compose up -d
docker ps
docker logs <container>
```

#### 2. Update Notebook Code

In the notebook cells, replace:
- `'podman-compose'` → `'docker-compose'`
- `'podman'` → `'docker'`

#### 3. Docker Compose File Compatibility

The `docker-compose.yml` file in this lab is compatible with both Podman and Docker. No changes needed.

#### 4. Volume Permissions

**Podman:** Runs rootless by default, so volume permissions match your user.

**Docker:** May require adjusting volume permissions:

```bash
# If you encounter permission errors with Docker
sudo chown -R $USER:$USER ./infrastructure/vault/data
```

#### 5. Network Differences

**Podman:** Uses CNI networking by default.

**Docker:** Uses bridge networking by default.

Both work fine for this lab, but if you have networking issues:

```bash
# Docker: Check network
docker network ls
docker network inspect agentic-security-incubation_default

# Podman: Check network
podman network ls
podman network inspect agentic-security-incubation_default
```

### When to Use Which?

**Use Podman if:**
- You want rootless containers (better security)
- You're on Linux and want daemonless operation
- You need Kubernetes pod compatibility
- You prefer open-source without vendor lock-in

**Use Docker if:**
- You're already familiar with Docker
- You need Docker Desktop features (GUI, Kubernetes)
- You're on Windows/Mac and want native integration
- Your team standardizes on Docker

### Testing Docker Compatibility

In [ ]:
print("🔍 Checking Docker installation...\n")

# Check Docker
try:
    result = subprocess.run(['docker', '--version'], capture_output=True, text=True, check=True)
    print(f"✅ Docker: {result.stdout.strip()}")
    
    # Check Docker Compose
    result = subprocess.run(['docker-compose', '--version'], capture_output=True, text=True, check=True)
    print(f"✅ Docker Compose: {result.stdout.strip()}")
    
    print("\n💡 Docker is installed. You can use Docker instead of Podman.")
    print("   Just replace 'podman' with 'docker' in all commands.")
    
except (subprocess.CalledProcessError, FileNotFoundError):
    print("❌ Docker not found.")
    print("   Install from: https://docs.docker.com/get-docker/")

print("\n" + "="*80)

# Check Podman
try:
    result = subprocess.run(['podman', '--version'], capture_output=True, text=True, check=True)
    print(f"✅ Podman: {result.stdout.strip()}")
    
    # Check Podman Compose
    result = subprocess.run(['podman-compose', '--version'], capture_output=True, text=True, check=True)
    print(f"✅ Podman Compose: {result.stdout.strip()}")
    
    print("\n💡 Podman is installed and ready to use.")
    
except (subprocess.CalledProcessError, FileNotFoundError):
    print("❌ Podman not found.")
    print("   Install from: https://podman.io/getting-started/installation")

---

## ✅ Summary

This notebook covered:

### Testing
- ✅ Service status checks
- ✅ Connectivity tests for all components
- ✅ Vault configuration verification

### Troubleshooting
- ✅ Log inspection commands
- ✅ Service restart procedures
- ✅ Common issues and solutions

### Cleanup
- ✅ Stopping services (with/without data)
- ✅ Terraform destroy
- ✅ Image removal

### Docker vs Podman
- ✅ Key differences
- ✅ Migration guide
- ✅ When to use which

---

**🎉 Lab Complete!**

You now have all the tools to test, troubleshoot, and clean up your Agentic Security lab environment.